# Prediksi Diabetes Menggunakan Machine Learning
**Tugas PBL – Mata Kuliah FDA**

Kelompok:
1. Gemma Erica Putra (19250465)
2. Ardiansyah (19252254)
3. Najma Alyssa Yasmina (19252566)
4. Nathasya Aprilya Putri (19251072)
5. Icha Dwi Kusuma (19250543)
6. Arya Permana Putra (19251328)

Notebook ini berisi seluruh proses analisis dari awal sampai akhir dengan alur **CRISP-DM**:

1. Business Understanding
2. Data Understanding
3. Data Preparation
4. Modelling
5. Evaluation

**Cara menjalankan di Google Colab:** klik menu *Runtime → Run all*. Saat sel *Load Dataset* dijalankan, akan muncul tombol **Choose Files**; pilih file `diabetes_prediction_dataset.csv`.

Setiap sel kode diberi komentar `#` yang menjelaskan langkah demi langkah apa yang sedang dikerjakan.

## 1. Business Understanding

- **Masalah:** Diabetes sering terlambat terdeteksi. Screening manual membutuhkan banyak waktu dan biaya.
- **Tujuan:** Membangun model klasifikasi yang memprediksi apakah seorang pasien **menderita diabetes (1)** atau **tidak (0)** berdasarkan data demografi dan klinis.
- **Dataset:** *Diabetes Prediction Dataset* (100.000 baris, 9 kolom) dengan 8 fitur dan 1 target (`diabetes`).
- **Ukuran keberhasilan:** Karena kasus diabetes jauh lebih sedikit daripada non-diabetes, akurasi saja tidak cukup. Kita juga menilai **Recall, Precision, F1-score, dan ROC-AUC**.

## 2. Import Library

In [ ]:
# --- Library untuk manipulasi data ---
import numpy as np              # operasi numerik
import pandas as pd             # tabel data (DataFrame)

# --- Library untuk visualisasi ---
import matplotlib.pyplot as plt # grafik dasar
import seaborn as sns           # grafik statistik yang lebih rapi

# --- Library machine learning (scikit-learn) ---
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             roc_auc_score, roc_curve, confusion_matrix,
                             ConfusionMatrixDisplay, classification_report)
from sklearn.inspection import permutation_importance

import warnings
warnings.filterwarnings("ignore")   # sembunyikan peringatan agar output bersih

# Pengaturan tampilan grafik
sns.set_theme(style="whitegrid")
plt.rcParams["figure.dpi"] = 110

# Seed agar hasil selalu sama setiap dijalankan (reproducible)
SEED = 42

## 3. Load Dataset

In [ ]:
# Langkah 1: unggah file CSV.
# Jika dijalankan di Google Colab, akan muncul tombol "Choose Files".
# Jika dijalankan di luar Colab (Jupyter lokal), file dibaca langsung dari folder yang sama.
try:
    from google.colab import files
    uploaded = files.upload()                 # dialog unggah file
    nama_file = list(uploaded.keys())[0]      # ambil nama file yang diunggah
except ImportError:
    nama_file = "diabetes_prediction_dataset.csv"

# Langkah 2: baca file CSV menjadi DataFrame
df_raw = pd.read_csv(nama_file)

# Langkah 3: simpan salinan agar data asli tidak berubah saat diproses
df = df_raw.copy()
print("Jumlah baris dan kolom:", df.shape)

## 4. Data Understanding
Tahap ini bertujuan mengenali data dan menemukan potensi masalah: *missing value*, duplikat, outlier, distribusi tidak seimbang, dan kategori yang ambigu.

### 4.1 Melihat struktur data

In [ ]:
# Tampilkan 5 baris pertama untuk melihat contoh isi data
df.head()

In [ ]:
# Info tipe data setiap kolom dan jumlah nilai non-null
df.info()

In [ ]:
# Statistik deskriptif untuk kolom numerik
# (count, mean, std, min, kuartil, max)
df.describe().T

In [ ]:
# Ringkasan untuk kolom kategorikal
df.describe(include="object").T

### 4.2 Memeriksa missing value dan duplikat

In [ ]:
# Hitung jumlah nilai kosong (NaN) di setiap kolom
print("Missing value per kolom:")
print(df.isnull().sum())

# Hitung jumlah baris yang duplikat (seluruh kolom sama persis)
jumlah_duplikat = df.duplicated().sum()
print("\nJumlah baris duplikat:", jumlah_duplikat)
print("Persentase duplikat  :", round(jumlah_duplikat / len(df) * 100, 2), "%")

In [ ]:
# Periksa isi kolom kategorikal untuk mencari kategori yang ambigu atau jarang
for kolom in ["gender", "smoking_history"]:
    print(f"--- {kolom} ---")
    print(df[kolom].value_counts(), "\n")

### 4.3 Distribusi variabel target

In [ ]:
# Hitung jumlah dan persentase tiap kelas target (0 = tidak diabetes, 1 = diabetes)
hitung = df["diabetes"].value_counts()
persen = df["diabetes"].value_counts(normalize=True) * 100
print(pd.DataFrame({"jumlah": hitung, "persen": persen.round(2)}))

# Visualisasi diagram batang
fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(x="diabetes", data=df, palette=["#4C72B0", "#C44E52"], ax=ax)
for p in ax.patches:                                   # tulis angka di atas batang
    ax.annotate(f"{int(p.get_height()):,}", (p.get_x() + p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
ax.set_xticklabels(["Tidak Diabetes (0)", "Diabetes (1)"])
ax.set_xlabel("")
ax.set_ylabel("Jumlah pasien")
plt.tight_layout()
plt.show()
# Kesimpulan: data tidak seimbang (imbalanced) -> perlu penanganan khusus saat modelling

### 4.4 Distribusi fitur numerik

In [ ]:
# Histogram untuk melihat bentuk sebaran 4 fitur numerik utama
fitur_num = ["age", "bmi", "HbA1c_level", "blood_glucose_level"]
fig, axes = plt.subplots(2, 2, figsize=(10, 7))
for ax, kolom in zip(axes.ravel(), fitur_num):
    sns.histplot(df[kolom], bins=40, kde=True, color="#4C72B0", ax=ax)
    ax.set_title(kolom)
    ax.set_xlabel("")
plt.tight_layout()
plt.show()
# Perhatikan: bmi memiliki "spike" (menumpuk) di sekitar 27.32,
# HbA1c dan glukosa hanya memiliki nilai diskret tertentu.

In [ ]:
# Cek spike pada BMI: nilai 27.32 muncul sangat sering
print("Nilai BMI paling sering muncul:")
print(df["bmi"].value_counts().head(5))

### 4.5 Mendeteksi outlier

In [ ]:
# Boxplot untuk melihat titik di luar batas wajar (outlier)
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, kolom in zip(axes, fitur_num):
    sns.boxplot(y=df[kolom], color="#8172B3", ax=ax)
    ax.set_title(kolom)
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Hitung jumlah outlier dengan aturan IQR:
# outlier jika nilai < Q1 - 1.5*IQR  atau  nilai > Q3 + 1.5*IQR
def hitung_outlier(seri):
    q1, q3 = seri.quantile(0.25), seri.quantile(0.75)
    iqr = q3 - q1
    batas_bawah, batas_atas = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    n = ((seri < batas_bawah) | (seri > batas_atas)).sum()
    return batas_bawah, batas_atas, n

baris = []
for kolom in fitur_num:
    bb, ba, n = hitung_outlier(df[kolom])
    baris.append([kolom, round(bb, 2), round(ba, 2), n, round(n / len(df) * 100, 2)])
tabel_outlier = pd.DataFrame(baris, columns=["fitur", "batas_bawah", "batas_atas", "jumlah_outlier", "persen"])
tabel_outlier

### 4.6 Hubungan fitur dengan diabetes

In [ ]:
# Persentase penderita diabetes pada tiap kategori (gender, smoking_history, hipertensi, penyakit jantung)
fig, axes = plt.subplots(2, 2, figsize=(11, 7))
for ax, kolom in zip(axes.ravel(), ["gender", "smoking_history", "hypertension", "heart_disease"]):
    laju = df.groupby(kolom)["diabetes"].mean() * 100     # rata-rata target x 100 = % diabetes
    laju.plot(kind="bar", color="#C44E52", ax=ax)
    ax.set_title(f"% Diabetes berdasarkan {kolom}")
    ax.set_ylabel("% diabetes")
    ax.set_xlabel("")
    ax.tick_params(axis="x", rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot fitur numerik dipisah menurut kelas diabetes
fig, axes = plt.subplots(1, 4, figsize=(14, 4))
for ax, kolom in zip(axes, fitur_num):
    sns.boxplot(x="diabetes", y=kolom, data=df, palette=["#4C72B0", "#C44E52"], ax=ax)
    ax.set_title(kolom)
    ax.set_xlabel("diabetes")
    ax.set_ylabel("")
plt.tight_layout()
plt.show()

In [ ]:
# Heatmap korelasi antar variabel numerik (termasuk target)
korelasi = df.select_dtypes(include="number").corr()
plt.figure(figsize=(7, 5.5))
sns.heatmap(korelasi, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.tight_layout()
plt.show()
print(korelasi["diabetes"].drop("diabetes").sort_values(ascending=False).round(3))

## 5. Data Preparation
Masalah yang ditemukan pada tahap *Data Understanding* ditangani satu per satu:

| Masalah | Penanganan |
|---|---|
| Baris duplikat | Dihapus |
| Kategori `smoking_history` ambigu (`ever`, `former`, `not current`) | Digabung menjadi `former` |
| Outlier `bmi` | Dibatasi dengan *clipping* IQR |
| Fitur kategorikal | *One-hot encoding* |
| Skala fitur berbeda | *Standardization* (khusus Logistic Regression) |
| Data tidak seimbang | `class_weight="balanced"` + *stratified split* |

In [ ]:
# Langkah 1: hapus baris duplikat
sebelum = len(df)
df = df.drop_duplicates().reset_index(drop=True)
print(f"Baris sebelum: {sebelum:,} | sesudah: {len(df):,} | dihapus: {sebelum - len(df):,}")

In [ ]:
# Langkah 2: sederhanakan kategori smoking_history
# 'ever', 'former', dan 'not current' sama-sama berarti sudah/pernah merokok tetapi tidak sedang merokok
peta_rokok = {
    "never": "never",
    "No Info": "no_info",
    "current": "current",
    "former": "former",
    "ever": "former",
    "not current": "former",
}
df["smoking_history"] = df["smoking_history"].map(peta_rokok)
print(df["smoking_history"].value_counts())

In [ ]:
# Langkah 3: tangani outlier BMI dengan clipping (winsorizing) memakai batas IQR
# Outlier tidak dibuang agar tidak kehilangan data; nilainya hanya dibatasi pada batas wajar.
# Catatan: HbA1c dan glukosa darah TIDAK di-clip karena nilai tingginya justru
# merupakan indikator klinis penting diabetes.
bb, ba, n = hitung_outlier(df["bmi"])
print(f"Batas BMI: {bb:.2f} - {ba:.2f} | outlier yang di-clip: {n:,}")
df["bmi"] = df["bmi"].clip(lower=bb, upper=ba)
print("BMI setelah clipping -> min:", df["bmi"].min().round(2), "| max:", df["bmi"].max().round(2))

In [ ]:
# Langkah 4: one-hot encoding untuk kolom kategorikal
# drop_first=True menghindari multikolinearitas (dummy variable trap)
df_model = pd.get_dummies(df, columns=["gender", "smoking_history"], drop_first=True, dtype=int)
print("Jumlah kolom setelah encoding:", df_model.shape[1])
df_model.head()

In [ ]:
# Langkah 5: pisahkan fitur (X) dan target (y)
X = df_model.drop(columns="diabetes")
y = df_model["diabetes"]

# Langkah 6: bagi data 80% training dan 20% testing
# stratify=y menjaga proporsi diabetes/non-diabetes tetap sama di kedua bagian
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED)

print("Data training:", X_train.shape, "| Data testing:", X_test.shape)
print("Proporsi diabetes (train):", round(y_train.mean() * 100, 2), "%")
print("Proporsi diabetes (test) :", round(y_test.mean() * 100, 2), "%")

## 6. Modelling
Dua algoritma klasifikasi dibandingkan:

1. **Logistic Regression** – model linear, mudah diinterpretasi (baseline)
2. **Gradient Boosting** (`HistGradientBoostingClassifier`) – pohon dibangun berurutan, masing-masing memperbaiki kesalahan pohon sebelumnya

In [ ]:
# Definisi model.
# class_weight="balanced" memberi bobot lebih besar pada kelas minoritas (diabetes)
# agar model tidak hanya "menebak tidak diabetes" untuk semua pasien.
daftar_model = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),                 # skala fitur harus seragam untuk model linear
        ("clf", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=SEED))
    ]),
    "Gradient Boosting": HistGradientBoostingClassifier(
        class_weight="balanced", random_state=SEED),
}

In [ ]:
# Cross-validation 5-fold (stratified) pada DATA TRAINING saja.
# Tujuannya memeriksa kestabilan model sebelum diuji pada data testing.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
metrik = ["accuracy", "precision", "recall", "f1", "roc_auc"]

hasil_cv = {}
for nama, model in daftar_model.items():
    skor = cross_validate(model, X_train, y_train, cv=cv, scoring=metrik, n_jobs=-1)
    hasil_cv[nama] = {m: skor[f"test_{m}"].mean() for m in metrik}
    print(f"{nama:<20} selesai")

tabel_cv = pd.DataFrame(hasil_cv).T.round(4)
tabel_cv

## 7. Evaluation
Kedua model dilatih pada seluruh data training lalu **diuji pada data testing** yang belum pernah dilihat model.

In [ ]:
# Kedua model yang dibandingkan
semua_model = daftar_model

baris, prob_test = [], {}
for nama, model in semua_model.items():
    model.fit(X_train, y_train)                         # latih pada data training
    pred = model.predict(X_test)                        # prediksi kelas pada data testing
    prob = model.predict_proba(X_test)[:, 1]            # probabilitas kelas diabetes
    prob_test[nama] = prob
    baris.append([nama,
                  accuracy_score(y_test, pred), precision_score(y_test, pred),
                  recall_score(y_test, pred), f1_score(y_test, pred),
                  roc_auc_score(y_test, prob)])

hasil_test = pd.DataFrame(baris, columns=["Model", "Accuracy", "Precision", "Recall", "F1", "ROC-AUC"])
hasil_test = hasil_test.set_index("Model").round(4)
hasil_test

In [ ]:
# Grafik perbandingan metrik antar model
hasil_test[["Accuracy", "Precision", "Recall", "F1", "ROC-AUC"]].plot(
    kind="bar", figsize=(8, 5), colormap="viridis", edgecolor="white")
plt.ylim(0.3, 1.02)
plt.ylabel("Skor")
plt.xlabel("")
plt.xticks(rotation=15)
plt.grid(axis="x", visible=False)   # hilangkan garis grid vertikal
plt.legend(loc="upper center", bbox_to_anchor=(0.5, -0.2), ncol=5)  # legenda di bawah grafik
plt.tight_layout()
plt.show()

In [ ]:
# Kurva ROC: semakin dekat ke pojok kiri atas, semakin baik model membedakan kelas
plt.figure(figsize=(6.5, 5.5))
for nama, prob in prob_test.items():
    fpr, tpr, _ = roc_curve(y_test, prob)
    plt.plot(fpr, tpr, label=f"{nama} (AUC={roc_auc_score(y_test, prob):.3f})")
plt.plot([0, 1], [0, 1], "k--", label="Acak (AUC=0.500)")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate (Recall)")
plt.legend(loc="lower right", fontsize=8)
plt.tight_layout()
plt.show()

### 7.1 Model terbaik: confusion matrix dan laporan klasifikasi

In [ ]:
# Model final dipilih berdasarkan F1 cross-validation pada data training
# (bukan berdasarkan data testing, agar penilaian tetap adil).
nama_final = tabel_cv["f1"].idxmax()
model_final = semua_model[nama_final]
pred_final = model_final.predict(X_test)
print("MODEL FINAL:", nama_final, "\n")

# Laporan klasifikasi lengkap (precision, recall, f1 per kelas)
print(classification_report(y_test, pred_final, target_names=["Tidak Diabetes", "Diabetes"], digits=4))

# Confusion matrix
fig, ax = plt.subplots(figsize=(5.5, 4.5))
ConfusionMatrixDisplay(confusion_matrix(y_test, pred_final),
                       display_labels=["Tidak Diabetes", "Diabetes"]).plot(
    cmap="Blues", values_format="d", ax=ax, colorbar=False)
plt.tight_layout()
plt.show()

tn, fp, fn, tp = confusion_matrix(y_test, pred_final).ravel()
print(f"TN={tn:,} | FP={fp:,} | FN={fn:,} | TP={tp:,}")

### 7.2 Fitur yang paling berpengaruh

In [ ]:
# Permutation importance: seberapa turun F1 jika sebuah fitur diacak.
# Semakin besar penurunan, semakin penting fitur tersebut.
hasil_imp = permutation_importance(model_final, X_test, y_test, scoring="f1",
                                   n_repeats=5, random_state=SEED, n_jobs=-1)
importance = pd.Series(hasil_imp.importances_mean, index=X_test.columns).sort_values()

plt.figure(figsize=(7, 5))
importance.plot(kind="barh", color="#55A868")
plt.xlabel("Penurunan F1-score")
plt.tight_layout()
plt.show()
print(importance.sort_values(ascending=False).round(4))

### 7.3 Analisis ambang batas (threshold)

In [ ]:
# Secara default model mengklasifikasikan "diabetes" jika probabilitas >= 0.5.
# Dalam skrining kesehatan, kita sering ingin Recall lebih tinggi (jangan sampai pasien terlewat),
# sehingga ambang batas dapat diturunkan. Berikut dampaknya:
prob_final = prob_test[nama_final]
baris = []
for t in [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]:
    pred_t = (prob_final >= t).astype(int)
    baris.append([t, precision_score(y_test, pred_t), recall_score(y_test, pred_t), f1_score(y_test, pred_t)])
tabel_threshold = pd.DataFrame(baris, columns=["Threshold", "Precision", "Recall", "F1"]).round(4)
tabel_threshold

## 8. Kesimpulan

1. Data memiliki **ketidakseimbangan kelas** (sekitar 8,5% diabetes), ribuan baris duplikat, outlier BMI, dan kategori merokok yang ambigu. Semuanya ditangani pada tahap *Data Preparation*.
2. Fitur dengan hubungan terkuat terhadap diabetes adalah **HbA1c** dan **kadar glukosa darah**.
3. Model terbaik dan perbandingan Logistic Regression vs Gradient Boosting dapat dilihat pada tabel evaluasi dan sel *Model Final*.
4. **Catatan penting:** HbA1c dan glukosa darah adalah kriteria diagnostik diabetes itu sendiri, sehingga performa tinggi sebagian disebabkan fitur tersebut sangat informatif. Model ini cocok sebagai alat bantu skrining, **bukan pengganti diagnosis dokter**.